<a href="https://colab.research.google.com/github/amiralito/BindCraft2_colab/blob/main/BindCraft2_design_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/amiralito/BindCraft2_Colab/blob/main/BindCraft2_design_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BindCraft 2 design pipeline

End-to-end **BindCraft 2 (BC2)** binder design on Colab — [PacesaLab/BindCraft2](https://github.com/PacesaLab/BindCraft2).

**Pipeline** — AlphaFold2 gradient design (hallucination) -> ProteinMPNN redesign -> re-prediction
with held-out AF2 models -> filters -> ranking by `i_pDAE`.

Every design modality is exposed through the form: de novo `binder`, `large_binder`, `peptide`,
`cyclic_peptide`, `homo_oligomer`, `multidomain`, `VHH`, `ARP`, `scFv`, `Fab`, plus the conformational
objectives `induced_fit` and `fold_switch`. Design properties (`humanize`, `forced_targeting`,
`protease_stable`, `disulfide_staple`, `mixed_topology`, `termini_together`, `termini_accessible`,
`initial_guess`, `bigbang`), multi-target campaigns and off-target detargeting are all form fields,
and step 6b is a raw-JSON escape hatch for anything the form does not reach.

**Before you run anything**

- `Runtime -> Change runtime type -> GPU`. **BC2 does not run on CPU** — there is no CPU install path.
  A **T4 (16 GB) is the floor**; an **L4 / A100** is much more comfortable, and a card with compute
  capability >= 8.0 gets native bfloat16.
- **Keep the target small.** A worker is budgeted `2.0 x (3.4 GB + 38 kB x N^2)` for a padded complex
  of N residues. A 16 GB T4 fits roughly **250 residues of target + binder**. Step 5 trims your input
  and step 4 prints the estimate before you commit.
- **Mount Drive.** AF2 parameters are 5.3 GB; cached on Drive they download once. Results on Drive
  mean a campaign killed by a Colab timeout **resumes** — just re-run step 7.
- A campaign runs until it has `number_of_final_designs` accepted binders, with **many attempts per
  accepted design**. Set `max_trajectories` so the run is bounded, and stop the run cell whenever
  you have seen enough — nothing is lost.

---

## 1 · Check the GPU

In [ ]:
#@title Check allocated GPU { display-mode: "form" }
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv || echo "No GPU - set Runtime -> Change runtime type -> GPU"


## Setup · Run name, output directory & Google Drive

Name this run (the **binder / campaign name**) and mount Drive. Each run gets its own timestamped
folder — `<BASE_DIR>/runs/<date-time>_<RUN_NAME>` — so runs never overwrite each other, and the
`WORK` variable every downstream step uses points at it automatically.

AF2 parameters and the JAX compiled-graph cache are stored once under `<BASE_DIR>/bc2_cache` and
reused by every future run (step 3), so you only download the 5.3 GB the first time.

To **resume a campaign** that timed out, set `RESUME_RUN` to that run folder's name (e.g.
`2026-09-21_1432_NbNRC2cap`) instead of leaving it empty — steps 6 and 7 then carry on in place
rather than starting a new folder.

In [ ]:
#@title Mount Drive, name this run & set output directory { display-mode: "form" }
import os, sys, time, json, shutil, datetime, subprocess
from pathlib import Path

MOUNT_DRIVE = True                            #@param {type:"boolean"}
BASE_DIR    = "/content/drive/MyDrive/BC2"    #@param {type:"string"}
RUN_NAME    = "binder"                        #@param {type:"string"}
RESUME_RUN  = ""                              #@param {type:"string"}
#  BASE_DIR holds the shared weight cache, a targets/ folder and a runs/ folder.
#  Each run gets its own timestamped sub-folder named from RUN_NAME.
#  RESUME_RUN: name of an existing runs/<folder> to carry on in. Leave empty for a new run.

RUNTIME = Path("/content") if os.access("/content", os.W_OK) else Path.home()

ON_DRIVE = False
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ON_DRIVE = True
    except Exception as problem:
        print(f"Drive did not mount: {problem}")
        print("Allow third-party cookies for colab.research.google.com, or mount it from the")
        print("folder icon on the left. Falling back to this runtime's disk.")

BASE = Path(BASE_DIR) if ON_DRIVE else (RUNTIME / "BC2")
RUNS_DIR   = BASE / "runs"
CACHE_DIR  = BASE / "bc2_cache"
TARGET_DIR = BASE / "targets"

if RESUME_RUN.strip():
    WORK = RUNS_DIR / RESUME_RUN.strip()
    if not WORK.exists():
        raise SystemExit(f"No such run: {WORK}\nLeave RESUME_RUN empty to start a new one.")
    RUN_NAME = WORK.name.split("_", 2)[-1] or RUN_NAME
else:
    stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
    WORK = RUNS_DIR / f"{stamp}_{RUN_NAME}"

for directory in (RUNS_DIR, CACHE_DIR, TARGET_DIR, WORK):
    directory.mkdir(parents=True, exist_ok=True)

# BC2 reads both of these at import time, so they are set before it is ever imported.
os.environ["BINDCRAFT_WEIGHTS"] = str(CACHE_DIR)
os.environ["JAX_COMPILATION_CACHE_DIR"] = str(CACHE_DIR / "compile_cache")

CAMPAIGN_JSON = WORK / "campaign.json"
PROJECT_FOLDER = WORK / "results"

print(f"run name : {RUN_NAME}")
print(f"WORK     : {WORK}" + ("  (resuming)" if RESUME_RUN.strip() else ""))
print(f"results  : {PROJECT_FOLDER}")
print(f"targets  : {TARGET_DIR}")
print(f"weights  : {CACHE_DIR}"
      + ("  (on Drive - here next session)" if ON_DRIVE else "  (this runtime - fetched again each session)"))
print(f"free disk: {shutil.disk_usage(RUNTIME).free / 1e9:.0f} GB   (the weights need ~11 GB while unpacking)")
if not ON_DRIVE:
    print("\nNOT on Drive: a Colab timeout loses this campaign and the 5.3 GB of weights.")


## 2 · Install BindCraft 2

Clones `PacesaLab/BindCraft2` and installs it **editable** into the Colab kernel — editable because
`settings/` and `scaffolds/` sit beside the package and are resolved relative to it, so the clone
*is* the installation.

The accelerator wheels are chosen the way `install.sh` chooses them: the driver's CUDA major version
picks `cuda13` or `cuda12`, and a card below compute capability 7.5 takes `cuda12` however new the
driver is. If the optional cuEquivariance ops wheel has no build for this runtime the cell retries
without it — those kernels are off unless a campaign asks for them.

This is the slow cell (a few minutes). It is idempotent: re-running it is cheap.

In [ ]:
#@title Install BindCraft 2 (~3-5 min the first time) { display-mode: "form" }
import re
ACCELERATOR_CHOICE = "auto"  #@param ["auto", "cuda13", "cuda12"]
BRANCH             = "main"  #@param {type:"string"}

t0 = time.time()
REPO = RUNTIME / "BindCraft2"
GITHUB = "https://github.com/PacesaLab/BindCraft2.git"

if sys.version_info < (3, 12):
    raise SystemExit(f"This runtime is Python {sys.version.split()[0]}; BC2 needs 3.12 or newer. "
                     "Pick a newer Colab runtime.")

def card_facts():
    """Some drivers do not answer every field. A half-described card is still a card."""
    for fields in ("name,memory.total,compute_cap", "name,memory.total", "name"):
        probe = subprocess.run(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader"],
                               capture_output=True, text=True)
        if probe.returncode == 0 and probe.stdout.strip():
            answered = [f.strip() for f in probe.stdout.splitlines()[0].split(",")]
            return answered + [""] * (3 - len(answered))
    return None

facts = card_facts()
if facts is None:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then run this cell again.")
GPU_NAME, gpu_memory, GPU_COMPUTE = facts
try:
    GPU_MEMORY_GB = float(gpu_memory.split()[0]) / 1024
except (IndexError, ValueError):
    GPU_MEMORY_GB = 0.0
print(f"GPU: {GPU_NAME}" + (f", {GPU_MEMORY_GB:.0f} GB" if GPU_MEMORY_GB else "")
      + (f", compute {GPU_COMPUTE}" if GPU_COMPUTE else ""))
if GPU_COMPUTE and float(GPU_COMPUTE) < 8.0:
    print("  no native bfloat16 on this card - designing is slower than on an L4 / A100")

def git(*arguments):
    done = subprocess.run(["git", *[str(a) for a in arguments]], capture_output=True, text=True,
                          env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if done.returncode != 0:
        print((done.stdout + done.stderr).strip())
    return done

if not (REPO / ".git").exists():
    shutil.rmtree(REPO, ignore_errors=True)
    if git("clone", "--quiet", GITHUB, REPO).returncode != 0:
        raise SystemExit(f"Could not clone {GITHUB}. Check the runtime's network.")
if git("-C", REPO, "fetch", "--quiet", "origin", BRANCH).returncode == 0:
    git("-C", REPO, "checkout", "--quiet", "FETCH_HEAD")
REVISION = subprocess.run(["git", "-C", str(REPO), "describe", "--always"],
                          capture_output=True, text=True).stdout.strip()
print(f"source: {BRANCH} ({REVISION})")

def driver_cuda_major():
    listing = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else ""
    found = re.search(r"CUDA Version:\s*(\d+)", listing)
    return int(found.group(1)) if found else 0

if ACCELERATOR_CHOICE != "auto":
    ACCELERATOR = ACCELERATOR_CHOICE
else:
    ACCELERATOR = "cuda12" if driver_cuda_major() == 12 else "cuda13"
    if ACCELERATOR == "cuda13" and GPU_COMPUTE and float(GPU_COMPUTE) < 7.5:
        ACCELERATOR = "cuda12"   # CUDA 13 dropped everything below compute capability 7.5

PYTHON   = sys.executable
LAUNCHER = [PYTHON, "bindcraft.py"]
PIP_ENV  = {**os.environ, "PIP_BREAK_SYSTEM_PACKAGES": "1", "PIP_DISABLE_PIP_VERSION_CHECK": "1"}
PIP_NOISE = (r"already satisfied|^\s*(Collecting|Downloading|Using cached|Installing collected|"
             r"Successfully|Preparing|Building|Created wheel|Stored in|Attempting uninstall|"
             r"Found existing|Uninstalling|Obtaining|Checking|Getting|Installing build|Requirement|"
             r"WARNING|ERROR: pip|\s*\|)")

def stream(command, quiet_pattern=None, **extra):
    """Run a command and print it as it goes; lines matching quiet_pattern are kept but not shown."""
    process = subprocess.Popen([str(p) for p in command], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, **extra)
    kept = []
    for line in process.stdout:
        kept.append(line)
        if quiet_pattern is None or not re.search(quiet_pattern, line):
            print(line, end="")
    return process.wait(), "".join(kept)

print(f"installing for {ACCELERATOR} ...")
stream([PYTHON, "-m", "pip", "install", "-e", f".[{ACCELERATOR}]"],
       quiet_pattern=PIP_NOISE, cwd=REPO, env=PIP_ENV)

def package_is_sound():
    """selfcheck names every module the accelerator needs and the checkpoints inside the package.
    --shipped-only leaves the AlphaFold weights out of it; those come in step 3."""
    done = subprocess.run([PYTHON, "-m", "bindcraft.selfcheck", ACCELERATOR, "--shipped-only"],
                          cwd=REPO, capture_output=True, text=True)
    return done.returncode == 0, (done.stdout + done.stderr).strip()

sound, complaint = package_is_sound()
if not sound:
    # cuEquivariance's ops wheel is the piece most likely to have no build for a given runtime.
    # Those kernels are optional (use_cueq is off unless a campaign asks), so plain jax is complete.
    print(f"{complaint}\nretrying without the optional cuEquivariance kernels")
    subprocess.run([PYTHON, "-m", "pip", "install", "--quiet", "-e", "."], cwd=REPO, env=PIP_ENV, check=False)
    subprocess.run([PYTHON, "-m", "pip", "install", "--quiet", f"jax[{ACCELERATOR}]>=0.11,<0.12"],
                   cwd=REPO, env=PIP_ENV, check=False)
    sound, complaint = package_is_sound()
if not sound:
    raise SystemExit(f"BC2 is not installed. Missing:\n{complaint}\n\n"
                     f"Python {sys.version.split()[0]} with {ACCELERATOR}. "
                     f"Try ACCELERATOR_CHOICE = {'cuda12' if ACCELERATOR != 'cuda12' else 'cuda13'}.")
print(f"package: ready ({ACCELERATOR})")

# Light deps for the analysis cells below.
subprocess.run([PYTHON, "-m", "pip", "install", "--quiet", "gemmi", "py3Dmol", "pandas"],
               env=PIP_ENV, check=False)

def bindcraft(*arguments, quiet=False, capture=True):
    """Run a bindcraft subcommand through the repo launcher."""
    done = subprocess.run([*LAUNCHER, *[str(a) for a in arguments]], cwd=REPO, env=os.environ,
                          capture_output=capture, text=True)
    if capture and not quiet:
        print((done.stdout + done.stderr).rstrip())
    return done

SHIPPED_TARGETS    = sorted(p.stem for p in (REPO / "settings" / "target").glob("*.json"))
SHIPPED_MODALITIES = sorted(p.stem for p in (REPO / "settings" / "modality").glob("*.json"))
SHIPPED_PROPERTIES = sorted(p.stem for p in (REPO / "settings" / "property").glob("*.json"))
print(f"\nshipped targets   : {', '.join(SHIPPED_TARGETS)}")
print(f"shipped modalities: {', '.join(SHIPPED_MODALITIES)}")
print(f"shipped properties: {', '.join(SHIPPED_PROPERTIES)}")
print(f"\ninstall finished in {time.time() - t0:.0f}s")


## 3 · AlphaFold 2 parameters  *(once)*

Downloads the 5.3 GB parameter archive into the shared cache. A campaign designs against five
multimer models and holds two monomer models back to validate on, so a design is never scored by a
model that shaped it — all seven come from this one archive.

The cell checks whether the parameters are already there and **skips the download if so**, so with
Drive mounted every session after the first is instant. It then pins `BINDCRAFT_AF2_PARAMS`, which
overrides every other lookup, so no later step can decide they are missing and fetch them again.

In [ ]:
#@title Fetch AF2 parameters (once; cached for future sessions) { display-mode: "form" }
def parameter_directory():
    """Where the weights are, if they are here. BC2 accepts either layout."""
    return next((d for d in (CACHE_DIR / "alphafold", CACHE_DIR / "alphafold" / "params")
                 if any(d.glob("params_*.npz"))), None)

if parameter_directory() is None:
    print("weights: downloading 5.3 GB - a few minutes")
    status, _ = stream([*LAUNCHER, "fetch-weights"], quiet_pattern=r"of 5\.3 GB", cwd=REPO, env=os.environ)
    if status != 0:
        raise SystemExit("The weights did not arrive. Run this cell again - an interrupted download "
                         "is re-fetched. If it ran out of space, start a fresh runtime.")
else:
    print("weights: already cached - skipping download")

PARAMETERS = parameter_directory()
if PARAMETERS is None:
    raise SystemExit(f"No params_*.npz under {CACHE_DIR / 'alphafold'}. Run this cell again.")
os.environ["BINDCRAFT_AF2_PARAMS"] = str(PARAMETERS)
print(f"weights: ready, {len(list(PARAMETERS.glob('params_*.npz')))} checkpoints in {PARAMETERS}")

backend = subprocess.run([PYTHON, "-c", "import jax; print(jax.default_backend())"],
                         cwd=REPO, capture_output=True, text=True).stdout.strip()
print(f"jax    : {backend or 'did not start'}")
if backend != "gpu":
    print("  JAX did not take the GPU. Restart the runtime and re-run steps 2-3.")


## 4 · Add target(s)

**Run this cell once per target.** Leave `RESET_TARGETS` ticked for the first one and untick it to
add more — that is how you build a multi-target campaign (one binder, several targets) or add an
**off-target** to detarget against (`OBJECTIVE = "detarget"`).

| Source | What to set |
| --- | --- |
| `shipped` | `SHIPPED_NAME` — one of the targets BC2 ships, with its own structure and hotspots already selected |
| `pdb_id` | `PDB_ID` — fetched from RCSB into `<BASE_DIR>/targets/` |
| `upload` | pick a `.pdb` / `.cif` / `.fasta` from your machine |
| `path` | `TARGET_PATH` — a file already on Drive or in this runtime |

- **`CHAINS`** — which chains of the file are the target, e.g. `A` or `A,B` for a receptor assembly.
  Empty keeps every chain.
- **`HOTSPOTS`** — residues to bind, in the *input file's* numbering: `54,56,66-70`, or chain-prefixed
  `A54,B12-16` for a multi-chain target. Empty lets BC2 choose the epitope.
- **`COLDSPOTS`** — residues to keep clear.
- **FASTA targets** are peptides / disordered targets; pair them with the `peptide`, `cyclic_peptide`
  or `binder` modality and set `crop_fasta_sequence` in step 6 if you want a fixed window.

The cell prints residues per chain and the **memory estimate** for the padded complex — check it
against your card before going further.

In [ ]:
#@title Add a target { display-mode: "form" }
import urllib.request

RESET_TARGETS = True          #@param {type:"boolean"}
SOURCE        = "shipped"     #@param ["shipped", "pdb_id", "upload", "path"]
SHIPPED_NAME  = "hPDL1"       #@param {type:"string"}
PDB_ID        = "9FP6"        #@param {type:"string"}
TARGET_PATH   = ""            #@param {type:"string"}
TARGET_NAME   = ""            #@param {type:"string"}
CHAINS        = ""            #@param {type:"string"}
HOTSPOTS      = ""            #@param {type:"string"}
COLDSPOTS     = ""            #@param {type:"string"}
OBJECTIVE     = "bind"        #@param ["bind", "detarget"]
WEIGHT        = 1.0           #@param {type:"number"}
#  TARGET_NAME empty = named after the file / PDB ID / shipped preset.
#  CHAINS empty = keep every chain (and, for a shipped target, keep the preset's own selection).
#  HOTSPOTS / COLDSPOTS use the INPUT file's residue numbering. Chain-prefix them
#  ("A54,B12-16") when the target spans several chains.

if "REPO" not in globals():
    raise SystemExit("Run steps 1-3 first.")
if RESET_TARGETS or "TARGETS" not in globals():
    TARGETS = []

SEQUENCE_SUFFIXES = (".fasta", ".fa", ".faa")

def chain_residues(path):
    """Residues per chain, or per FASTA record, read with the biotite BC2 installed."""
    path = Path(path)
    if path.suffix.lower() in SEQUENCE_SUFFIXES:
        from biotite.sequence.io.fasta import FastaFile, get_sequences
        return {name.split()[0]: len(seq) for name, seq in get_sequences(FastaFile.read(str(path))).items()}
    from biotite.structure.io import load_structure
    structure = load_structure(str(path), model=1)
    alpha = structure[structure.atom_name == "CA"]
    residues = {}
    for chain, residue in zip(alpha.chain_id, alpha.res_id):
        residues.setdefault(str(chain), set()).add(int(residue))
    return {chain: len(numbers) for chain, numbers in sorted(residues.items())}

entry = {}
if SOURCE == "shipped":
    preset = REPO / "settings" / "target" / f"{SHIPPED_NAME}.json"
    if not preset.exists():
        raise SystemExit(f"No shipped target {SHIPPED_NAME!r}. Available: {', '.join(SHIPPED_TARGETS)}")
    spec = json.loads(preset.read_text())
    entry = dict(spec["targets"][0])
    entry["target_path"] = str((preset.parent / entry["target_path"]).resolve())
    # The form only overrides what you actually filled in; the preset's own choices stand otherwise.
    if TARGET_NAME.strip(): entry["name"] = TARGET_NAME.strip()
    if CHAINS.strip():      entry["chains"] = CHAINS.strip()
    if HOTSPOTS.strip():    entry["hotspots"] = HOTSPOTS.strip()
    print(f"shipped target {SHIPPED_NAME}: {spec.get('description', '')}")
else:
    if SOURCE == "pdb_id":
        source_path = TARGET_DIR / f"{PDB_ID.upper()}.cif"
        if not source_path.exists():
            url = f"https://files.rcsb.org/download/{PDB_ID.upper()}.cif"
            print(f"fetching {url}")
            urllib.request.urlretrieve(url, source_path)
        default_name = PDB_ID.upper()
    elif SOURCE == "upload":
        from google.colab import files
        uploaded = files.upload()
        original = list(uploaded)[0]
        source_path = TARGET_DIR / original
        source_path.write_bytes(uploaded[original])
        default_name = Path(original).stem
    else:
        source_path = Path(TARGET_PATH).expanduser()
        if not source_path.exists():
            raise SystemExit(f"No such file: {source_path}")
        default_name = source_path.stem
    entry = {"name": (TARGET_NAME.strip() or default_name), "target_path": str(source_path.resolve())}
    if CHAINS.strip():   entry["chains"]   = CHAINS.strip()
    if HOTSPOTS.strip(): entry["hotspots"] = HOTSPOTS.strip()

if COLDSPOTS.strip():          entry["coldspots"] = COLDSPOTS.strip()
if OBJECTIVE == "detarget":    entry["objective"] = "detarget"
if abs(WEIGHT - 1.0) > 1e-9:   entry["weight"]    = float(WEIGHT)

TARGETS = [t for t in TARGETS if t["name"] != entry["name"]] + [entry]

print(f"\nadded: {entry['name']}  ({entry['target_path']})")
counts = chain_residues(entry["target_path"])
kept = [c.strip() for c in entry.get("chains", "").split(",") if c.strip()] or list(counts)
print("  chains in file :", ", ".join(f"{c}:{n}" for c, n in counts.items()))
print("  chains selected:", ", ".join(kept))
for key in ("hotspots", "coldspots", "objective", "weight"):
    if key in entry:
        print(f"  {key:<10}: {entry[key]}")

TARGET_RESIDUES = sum(n for c, n in counts.items() if c in kept)
print(f"\ncampaign targets ({len(TARGETS)}): "
      + ", ".join(f"{t['name']}{'*' if t.get('objective') == 'detarget' else ''}" for t in TARGETS)
      + "    (* = off-target)")
print(f"\nselected target is {TARGET_RESIDUES} residues.")

def memory_estimate(n_residues):
    """BC2 budgets 2.0 x (3.4 GB + 38 kB x N^2) per worker for a padded complex of N residues."""
    return 2.0 * (3.4 + 38e-6 * n_residues ** 2)

for binder in (60, 100, 150):
    total = TARGET_RESIDUES + binder
    print(f"  + {binder:>3} aa binder = {total:>4} residues -> ~{memory_estimate(total):5.1f} GB per worker")
if GPU_MEMORY_GB:
    headroom = GPU_MEMORY_GB - 4.0
    print(f"\nthis card has {GPU_MEMORY_GB:.0f} GB, ~{headroom:.0f} GB usable after BC2's 4 GB headroom.")
    if memory_estimate(TARGET_RESIDUES + 100) > headroom:
        print("  -> too big as it stands. Trim the target in step 5, or use a larger GPU.")


## 5 · Trim the target  *(do this for anything large)*

This is the cell that keeps BC2 in GPU memory. Cost scales as **O(N²)** in residues, so a full
deposited assembly with its ligands will OOM on a T4 and will still be slow on an A100. Trim to the
chains and residue window you actually design against, and drop non-polymer atoms (nucleotide, ions,
waters) unless you specifically need them.

Hotspots and coldspots are given in the **input file's** numbering, and trimming preserves residue
IDs, so the selections you set in step 4 stay valid after this cell.

Skip this cell if your target is already small and clean, or if it is a FASTA target.

In [ ]:
#@title Trim / clean a target { display-mode: "form" }
import numpy as np
import biotite.structure as struc
import biotite.structure.io as strucio

WHICH_TARGET     = ""      #@param {type:"string"}
KEEP_CHAINS      = ""      #@param {type:"string"}
RES_LO           = 0       #@param {type:"integer"}
RES_HI           = 0       #@param {type:"integer"}
STRIP_NONPOLYMER = True    #@param {type:"boolean"}
STRIP_HYDROGENS  = True    #@param {type:"boolean"}
#  WHICH_TARGET "" = the target added most recently. RES_LO/RES_HI both 0 = no residue trim.
#  KEEP_CHAINS "" = every chain currently selected.

if not globals().get("TARGETS"):
    raise SystemExit("Add a target in step 4 first.")

index  = next((i for i, t in enumerate(TARGETS) if t["name"] == WHICH_TARGET.strip()), len(TARGETS) - 1)
target = TARGETS[index]
source = Path(target["target_path"])
if source.suffix.lower() in SEQUENCE_SUFFIXES:
    raise SystemExit(f"{target['name']} is a FASTA target - nothing to trim. Use crop_fasta_sequence in step 6.")

structure = strucio.load_structure(str(source), model=1)
before = len(structure)

keep = np.ones(len(structure), dtype=bool)
chains = [c.strip() for c in (KEEP_CHAINS or target.get("chains", "")).split(",") if c.strip()]
if chains:
    keep &= np.isin(structure.chain_id, chains)
if STRIP_NONPOLYMER:
    keep &= struc.filter_amino_acids(structure)
if STRIP_HYDROGENS:
    keep &= structure.element != "H"
if RES_LO or RES_HI:
    lo = RES_LO if RES_LO else int(structure.res_id.min())
    hi = RES_HI if RES_HI else int(structure.res_id.max())
    keep &= (structure.res_id >= lo) & (structure.res_id <= hi)

trimmed = structure[keep]
if len(trimmed) == 0:
    raise SystemExit("Nothing left after trimming - check KEEP_CHAINS and the residue window.")
# Keep one altloc so the written file has one atom per name per residue.
if hasattr(trimmed, "altloc_id"):
    trimmed = trimmed[np.isin(trimmed.altloc_id, [".", "A", "", " "])]

out = TARGET_DIR / f"{target['name']}_trimmed.pdb"
strucio.save_structure(str(out), trimmed)

target["target_path"] = str(out.resolve())
if chains:
    target["chains"] = ",".join(chains)
TARGETS[index] = target

counts = chain_residues(out)
TARGET_RESIDUES = sum(counts.values())
print(f"{target['name']}: {before} -> {len(trimmed)} atoms")
print("  chains:", ", ".join(f"{c}:{n}" for c, n in counts.items()))
print("  residue range:", int(trimmed.res_id.min()), "-", int(trimmed.res_id.max()), "(numbering preserved)")
print("  written ->", out)
print(f"\n{TARGET_RESIDUES} target residues. "
      f"+100 aa binder -> ~{memory_estimate(TARGET_RESIDUES + 100):.1f} GB per worker.")


## 6 · Configure the campaign

Builds the campaign JSON that BC2 reads. Everything here is a documented setting; the layering is
**core -> modality -> properties -> target -> your campaign**, each layer overriding the one above,
and what this cell writes is the last layer, so it wins.

**Binder format** (`MODALITY`) — `binder` (de novo miniprotein), `large_binder` (>300 aa),
`peptide` (<25 aa linear), `cyclic_peptide` (head-to-tail), `homo_oligomer` (set `COPIES`),
`multidomain`, and the scaffolded formats `VHH`, `ARP`, `scFv`, `Fab` (these set their own lengths —
`BINDER_LENGTHS` is ignored). `EXTRA_MODALITY` adds a compatible conformational objective:
`induced_fit` (interface moves on binding) or `fold_switch` (whole fold changes).

**Properties** — tick what the experiment needs. `forced_targeting` requires named `HOTSPOTS` on a
structured target and also *requires* the accepted design to touch them. `humanize` applies human-like
sequence preferences plus an MHC-II anchor ceiling. `initial_guess` re-predicts each candidate from
the pose the trajectory folded (easier task, more permissive); `bigbang` seeds the gradient stages too.

**Budget** — `NUMBER_OF_FINAL_DESIGNS` is how many *accepted* binders to stop at; `MAX_TRAJECTORIES`
bounds the attempts spent getting there. On Colab, start with a handful of each. `TEST_RUN` opens
every gate and switches every filter off so the whole notebook runs through in minutes — **nothing it
accepts means anything**; use it only to check the plumbing.

`SEED` makes a campaign reproducible. `save_loss_plots` / `save_design_trajectory` are turned on so
step 9 has trajectory records to plot.

In [ ]:
#@title Build campaign.json { display-mode: "form" }
MODALITY                = "binder"   #@param ["binder", "large_binder", "peptide", "cyclic_peptide", "homo_oligomer", "multidomain", "VHH", "ARP", "scFv", "Fab"]
EXTRA_MODALITY          = "none"     #@param ["none", "induced_fit", "fold_switch"]
BINDER_LENGTHS          = "60-100"   #@param {type:"string"}
COPIES                  = 0          #@param {type:"integer"}
NUMBER_OF_FINAL_DESIGNS = 4          #@param {type:"integer"}
MAX_TRAJECTORIES        = 40         #@param {type:"integer"}
SEED                    = 0          #@param {type:"integer"}
#@markdown ---
#@markdown **Design properties**
forced_targeting   = False  #@param {type:"boolean"}
humanize           = False  #@param {type:"boolean"}
protease_stable    = False  #@param {type:"boolean"}
disulfide_staple   = False  #@param {type:"boolean"}
mixed_topology     = False  #@param {type:"boolean"}
termini_together   = False  #@param {type:"boolean"}
termini_accessible = False  #@param {type:"boolean"}
initial_guess      = False  #@param {type:"boolean"}
bigbang            = False  #@param {type:"boolean"}
#@markdown ---
#@markdown **Acceptance thresholds** (leave as-is unless you know why you are moving them)
MIN_IPTM_FINAL           = 0.6   #@param {type:"number"}
MIN_MONOMER_PLDDT_FINAL  = 0.7   #@param {type:"number"}
MAX_DETARGET_IPTM        = 0.5   #@param {type:"number"}
ALLOW_CYSTEINE           = False #@param {type:"boolean"}
CROP_FASTA_SEQUENCE      = ""    #@param {type:"string"}
CORE_PROFILE             = "none" #@param ["none", "benchmark"]
TEST_RUN                 = False #@param {type:"boolean"}
#  BINDER_LENGTHS: "80" one length, "60-100" a range, "55,65,75" a set.
#  COPIES: chains in a homo-oligomer (0 = not set). CROP_FASTA_SEQUENCE: "13-13" for a FASTA target.

if not globals().get("TARGETS"):
    raise SystemExit("Add at least one target in step 4 first.")

def parse_lengths(text):
    text = text.strip()
    if not text:
        return None
    if "-" in text:
        lo, hi = text.split("-", 1)
        return [int(lo), int(hi)]
    return [int(piece) for piece in text.split(",") if piece.strip()]

SCAFFOLDED = {"VHH", "ARP", "scFv", "Fab"}
modalities = [MODALITY] + ([EXTRA_MODALITY] if EXTRA_MODALITY != "none" else [])

campaign = {
    "campaign_name": RUN_NAME,
    "binder_name": RUN_NAME,
    "targets": TARGETS,
    "modality": modalities,
    "number_of_final_designs": int(NUMBER_OF_FINAL_DESIGNS),
    "project_folder": str(PROJECT_FOLDER),
    "resume": True,
    # Trajectory records, so step 9 has loss curves and the folded pose to compare against.
    "save_loss_plots": True,
    "save_design_trajectory": True,
    "save_binder_monomers": True,
}

lengths = parse_lengths(BINDER_LENGTHS)
if lengths and MODALITY not in SCAFFOLDED:
    campaign["binder_lengths"] = lengths
elif MODALITY in SCAFFOLDED:
    print(f"{MODALITY} is a scaffolded format - the scaffold sets the length, BINDER_LENGTHS ignored.")

if MAX_TRAJECTORIES > 0:        campaign["max_trajectories"] = int(MAX_TRAJECTORIES)
if SEED:                        campaign["campaign_seed"] = int(SEED)
if COPIES > 0:                  campaign["copies"] = int(COPIES)
if CORE_PROFILE != "none":      campaign["core"] = [CORE_PROFILE]
if CROP_FASTA_SEQUENCE.strip(): campaign["crop_fasta_sequence"] = parse_lengths(CROP_FASTA_SEQUENCE)
if not ALLOW_CYSTEINE and not disulfide_staple:
    campaign["aa_bias"] = {"C": 0}   # every shipped example does this unless disulfides are wanted

for name in ("forced_targeting", "humanize", "protease_stable", "disulfide_staple",
             "mixed_topology", "termini_together", "termini_accessible",
             "initial_guess", "bigbang"):
    if globals()[name]:
        campaign[name] = True

if not TEST_RUN:
    campaign.update({
        "min_plddt_refine": 0.65, "min_iptm_anneal": 0.5, "min_iptm_harden": 0.5,
        "min_plddt_mutate": 0.65, "min_iptm_mutate": 0.5,
        "min_iptm_final": float(MIN_IPTM_FINAL),
        "min_monomer_plddt_final": float(MIN_MONOMER_PLDDT_FINAL),
    })
    if MODALITY in ("peptide", "cyclic_peptide"):
        # A peptide need not have a confident free fold, so the monomer gate comes off.
        campaign.pop("min_monomer_plddt_final", None)
    if any(t.get("objective") == "detarget" for t in TARGETS):
        for stage in ("screen", "refine", "anneal", "harden", "mutate", "final"):
            campaign[f"max_detarget_iptm_{stage}"] = float(MAX_DETARGET_IPTM)
    if forced_targeting:
        campaign["min_hotspot_contact_final"] = 0.5
else:
    print("TEST_RUN: every gate open, every filter off. Nothing accepted here means anything.\n")

if forced_targeting and not any(t.get("hotspots") for t in TARGETS if t.get("objective") != "detarget"):
    print("forced_targeting is on but no target names hotspots - go back to step 4 and set them.\n")

CAMPAIGN_JSON.write_text(json.dumps(campaign, indent=2))
RAW_OVERRIDE = False
print(f"wrote {CAMPAIGN_JSON}\n")
print(json.dumps(campaign, indent=2))


## 6b (advanced) · Paste a raw campaign JSON  *(optional — overrides the form)*

Full manual control. Edit the JSON below and **run this cell instead of the form cell above** — it
writes exactly this to `campaign.json`, then continue to step 7.

This is where everything the form does not reach lives: per-loss weights (`weights_<name>`), the
`losses` block and its parameters, `filters` with explicit thresholds and directions, `binder_shapes`
for fold-switching states, `binder_scaffold` for your own scaffold, `parameter_sweep`, the multitarget
scheduling settings, `mpnn_variant`, `validation_models`, `design_recycles`, `desperation`, and so on.

`bindcraft design --list-settings` prints every accepted name; the same names with their defaults are
in `settings/core/reference.json`. The cell below prints both for reference before it writes.

**Paths**: `target_path` entries are resolved relative to this JSON file's directory, so absolute
paths (what steps 4–5 produce) are safest. `__TARGETS__` is substituted with the targets you built in
step 4, and `__PROJECT__` with this run's results folder — replace either with your own literal if
you want to bypass them.

In [ ]:
# ADVANCED: raw campaign JSON -> campaign.json  (edit freely; run this cell to override the form)
# Reference:
#   list(json.loads((REPO / "settings" / "core" / "reference.json").read_text())["settings"])
#   !{" ".join(map(str, LAUNCHER))} design --list-settings     # from the REPO directory

RAW_CAMPAIGN = r"""
{
  "campaign_name": "__RUN__",
  "binder_name": "__RUN__",
  "targets": __TARGETS__,
  "modality": ["binder"],

  "binder_lengths": [60, 100],
  "number_of_final_designs": 4,
  "max_trajectories": 40,

  "min_plddt_refine": 0.65,
  "min_iptm_anneal": 0.5,
  "min_iptm_harden": 0.5,
  "min_plddt_mutate": 0.65,
  "min_iptm_mutate": 0.5,
  "min_iptm_final": 0.6,
  "min_monomer_plddt_final": 0.7,
  "min_interface_buried_area_final": 500,

  "aa_bias": {"C": 0},

  "filters": {
    "Binder_RMSD": {"threshold": 3.5, "higher": false},
    "Surface_Hydrophobicity": {"threshold": 0.45, "higher": false}
  },

  "save_loss_plots": true,
  "save_design_trajectory": true,
  "save_binder_monomers": true,
  "resume": true,
  "project_folder": "__PROJECT__"
}
"""

RAW_CAMPAIGN = (RAW_CAMPAIGN
                .replace("__TARGETS__", json.dumps(globals().get("TARGETS", []), indent=2))
                .replace("__PROJECT__", str(PROJECT_FOLDER))
                .replace("__RUN__", RUN_NAME))
campaign = json.loads(RAW_CAMPAIGN)

ALLOWED = set(json.loads((REPO / "settings" / "core" / "reference.json").read_text())["settings"])
ALLOWED |= {"core", "modality", "targets", "target", "campaign_name", "binder_name",
            "paratope_conformations"} | set(SHIPPED_PROPERTIES)
unknown = [key for key in campaign if key not in ALLOWED]
if unknown:
    print("not recognised as settings (typo, or a newer name than this reference):", unknown, "\n")

CAMPAIGN_JSON.write_text(json.dumps(campaign, indent=2))
RAW_OVERRIDE = True
print(f"wrote {CAMPAIGN_JSON}  (raw override)\n")
print(json.dumps(campaign, indent=2))


## 7 · Run the campaign

Leave the cell running. **Each accepted design is written the moment it is found**, so the results
cells below have something to read long before the campaign finishes — and stopping the cell loses
nothing that was already written.

The campaign packs several design workers onto the card automatically (up to seven, as free memory
allows) because the card sits idle through ProteinMPNN redesign and validation. The first prediction
shape costs about a minute to compile; with the cache on Drive later runs read it back in seconds.

`resume` is on, so **re-running this cell carries on** in the same folder — which is what a Colab
timeout, or a second session, relies on. The log names each attempt, the stage it reached and any
filter a candidate failed; quiet stretches get a heartbeat line with the running counts.

In [ ]:
#@title Run the campaign { display-mode: "form" }
import csv, selectors

HEARTBEAT_SECONDS = 60   #@param {type:"integer"}

if not CAMPAIGN_JSON.exists():
    raise SystemExit("No campaign.json - run step 6 (or 6b) first.")
print(f"campaign : {CAMPAIGN_JSON}" + ("  (raw override)" if globals().get("RAW_OVERRIDE") else ""))
print(f"results  : {PROJECT_FOLDER}\n")

def rows_in(path):
    try:
        with open(path) as table:
            return max(0, sum(1 for _ in table) - 1)
    except OSError:
        return 0

def rejected_in(path):
    try:
        with open(path, newline="") as table:
            return sum(1 for row in csv.DictReader(table)
                       if (row.get("outcome") or "").strip() not in ("", "passed"))
    except OSError:
        return 0

started = time.time()

def status():
    minutes, seconds = divmod(int(time.time() - started), 60)
    attempts   = rows_in(PROJECT_FOLDER / "1_Trajectories" / "!_Trajectories.csv")
    candidates = rows_in(PROJECT_FOLDER / "2_Refolded" / "!_Refolded.csv")
    rejected   = rejected_in(PROJECT_FOLDER / "2_Refolded" / "!_Refolded.csv")
    accepted   = rows_in(PROJECT_FOLDER / "3_Ranked" / "!_Ranked.csv")
    return (f"[{minutes:>3}m{seconds:02d}]  attempts {attempts}  |  candidates {candidates} "
            f"({rejected} rejected)  |  accepted {accepted}")

process = subprocess.Popen([*LAUNCHER, "design", str(CAMPAIGN_JSON)],
                           cwd=REPO, env=os.environ, text=True, bufsize=1,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
selector = selectors.DefaultSelector()
selector.register(process.stdout, selectors.EVENT_READ)
try:
    while True:
        if selector.select(timeout=HEARTBEAT_SECONDS):
            line = process.stdout.readline()
            if not line:
                break
            print(line, end="")
        else:
            print(status(), flush=True)
    status_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    print("\nstopped. Everything already written is kept - re-run this cell to carry on.")
    status_code = None
finally:
    selector.close()

print("\n" + status())
if status_code == 0:
    print("campaign finished.")
elif status_code == 2:
    print("campaign refused - the reasons are printed above. Fix them in step 6 and run again.")
elif status_code:
    print(f"a design worker failed (exit {status_code}). "
          f"Check {PROJECT_FOLDER / 'workers'} for the per-worker logs.")


## 8 · Results table

`3_Ranked/!_Ranked.csv` is the single record of what was accepted, best first by **`i_pDAE`** —
distance-masked interface confidence, 0–1, higher is better. If nothing has been accepted yet, this
falls back to the candidate table and shows **which filter rejected each one**, which is the thing
worth reading when a campaign is not converging.

Key columns: `i_pDAE` / `i_pTM` interface confidence (0–1, higher better), `i_pAE` interface error
(lower better), `pLDDT` binder confidence, `Interface_Residues` interface size,
`Interface_BuriedArea` in Å², `Surface_Hydrophobicity` (high values may aggregate).
**None of these are affinities.**

In [ ]:
#@title Accepted designs, candidates and attempts { display-mode: "form" }
import pandas as pd
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

SHOW_ROWS = 20  #@param {type:"integer"}

CAMPAIGN_FOLDER = Path(PROJECT_FOLDER)
if not CAMPAIGN_FOLDER.exists():
    raise SystemExit(f"Nothing at {CAMPAIGN_FOLDER} yet - run step 7.")

def stage_table(*names):
    for relative in names:
        path = CAMPAIGN_FOLDER / relative
        if path.exists():
            return pd.read_csv(path), path
    return None, None

ACCEPTED,   accepted_path   = stage_table("3_Ranked/!_Ranked.csv", "ranked.csv", "accepted.csv")
CANDIDATES, candidate_path  = stage_table("2_Refolded/!_Refolded.csv", "candidates.csv")
ATTEMPTS,   attempt_path    = stage_table("1_Trajectories/!_Trajectories.csv", "trajectories.csv")

ACCEPTED_COLUMNS  = ["rank", "design", "Binder_Length", "i_pDAE", "i_pTM", "pLDDT", "i_pAE",
                     "Interface_Residues", "Interface_BuriedArea", "Surface_Hydrophobicity",
                     "Binder_Mass_kDa", "Binder_pI", "Binder_Sequence"]
CANDIDATE_COLUMNS = ["design", "outcome", "failed_filters", "i_pDAE", "i_pTM", "pLDDT", "i_pAE",
                     "Interface_Residues", "Binder_Sequence"]
ATTEMPT_COLUMNS   = ["design", "trajectory", "terminated", "i_pTM", "pLDDT"]

def show(frame, columns, title):
    if frame is None or not len(frame):
        return False
    present = [c for c in columns if c in frame.columns]
    print(f"\n=== {title}  ({len(frame)} rows) ===")
    display(frame[present].head(SHOW_ROWS) if present else frame.head(SHOW_ROWS))
    return True

if not show(ACCEPTED, ACCEPTED_COLUMNS, f"accepted, best first  [{accepted_path}]"):
    print("Nothing accepted yet.")
    if CANDIDATES is not None and len(CANDIDATES) and "failed_filters" in CANDIDATES.columns:
        reasons = (CANDIDATES["failed_filters"].dropna().astype(str)
                   .str.split(",").explode().str.strip())
        reasons = reasons[reasons != ""]
        if len(reasons):
            print("\nwhat is rejecting candidates:")
            for reason, count in reasons.value_counts().head(15).items():
                print(f"  {count:>4}  {reason}")
    if ATTEMPTS is not None and "terminated" in ATTEMPTS.columns:
        stopped = ATTEMPTS["terminated"].fillna("completed").replace("", "completed")
        print("\nwhere attempts stopped:")
        for stage, count in stopped.value_counts().items():
            print(f"  {count:>4}  {stage}")

show(CANDIDATES, CANDIDATE_COLUMNS, f"candidates  [{candidate_path}]")
show(ATTEMPTS,   ATTEMPT_COLUMNS,   f"attempts  [{attempt_path}]")


## 9 · Confidence plots

Three views of the campaign:

1. **Where the accepted designs sit** on the metrics that matter — `i_pDAE`, `i_pTM`, `pLDDT`,
   `Interface_Residues`, `Interface_BuriedArea`, `Surface_Hydrophobicity` — with the accepted
   designs marked against the full candidate distribution, so you can see whether the filters are
   cutting at a real shoulder or in the middle of a blob.
2. **Per-residue pLDDT** along the top design's complex, read from the mmCIF B-factor column
   (0–100 for BC2 predictions), with chain boundaries marked — the binder chain is the one to judge.
3. **The trajectory loss curve** for the top design, if `save_loss_plots` recorded one.

In [ ]:
#@title Plots { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np, gemmi

TOP_DESIGN = ""     #@param {type:"string"}
#  TOP_DESIGN "" = the best-ranked accepted design.

if ACCEPTED is None or not len(ACCEPTED):
    raise SystemExit("Nothing accepted yet - step 8 shows how far the campaign got.")

# --- 1. metric distributions -------------------------------------------------
METRICS = ["i_pDAE", "i_pTM", "pLDDT", "i_pAE",
           "Interface_Residues", "Interface_BuriedArea", "Surface_Hydrophobicity"]
available = [m for m in METRICS if m in ACCEPTED.columns]
if available:
    fig, axes = plt.subplots(1, len(available), figsize=(2.3 * len(available), 3.0))
    axes = np.atleast_1d(axes)
    for axis, metric in zip(axes, available):
        pool = (pd.to_numeric(CANDIDATES[metric], errors="coerce").dropna()
                if CANDIDATES is not None and metric in CANDIDATES.columns else pd.Series(dtype=float))
        kept = pd.to_numeric(ACCEPTED[metric], errors="coerce").dropna()
        if len(pool):
            axis.hist(pool, bins=20, color="#cfd6e0", edgecolor="none", label="candidates")
        for value in kept:
            axis.axvline(value, color="#c2456b", lw=1.2, alpha=0.9)
        axis.set_title(metric, fontsize=9)
        axis.tick_params(labelsize=7)
        axis.spines[["top", "right"]].set_visible(False)
    axes[0].set_ylabel("count", fontsize=8)
    fig.suptitle(f"{RUN_NAME}: accepted designs (rose) against all candidates (grey)", fontsize=10)
    fig.tight_layout()
    plt.show()

# --- 2. per-residue pLDDT of the top design ---------------------------------
ranked_dir = CAMPAIGN_FOLDER / "3_Ranked"
structures = [p for p in sorted(ranked_dir.glob("*.cif")) if "_monomer" not in p.name]
order = [str(d) for d in ACCEPTED.get("design", [])]
by_rank = [p for name in order for p in structures if p.stem.startswith(name)] or structures
chosen = next((p for p in by_rank if TOP_DESIGN.strip() and TOP_DESIGN.strip() in p.stem), None) or (by_rank[0] if by_rank else None)

if chosen is None:
    print("No accepted .cif to plot yet.")
else:
    st = gemmi.read_structure(str(chosen))
    st.setup_entities()
    model = st[0]
    values, boundaries, labels, position = [], [], [], 0
    for chain in model:
        residues = [r for r in chain if r.find_atom("CA", "*")]
        if not residues:
            continue
        for residue in residues:
            atom = residue.find_atom("CA", "*")
            values.append(atom.b_iso)
        position += len(residues)
        boundaries.append(position)
        labels.append((position - len(residues) / 2, chain.name, len(residues)))

    fig, axis = plt.subplots(figsize=(11, 3.1))
    axis.plot(range(1, len(values) + 1), values, lw=1.0, color="#2f5d8c")
    axis.fill_between(range(1, len(values) + 1), values, 0, color="#2f5d8c", alpha=0.12)
    for boundary in boundaries[:-1]:
        axis.axvline(boundary + 0.5, color="black", lw=0.8, ls="--", alpha=0.6)
    for centre, name, length in labels:
        axis.text(centre, 103, f"{name} ({length})", ha="center", fontsize=8)
    for level, colour in ((90, "#1f6f3f"), (70, "#c08a00"), (50, "#b03030")):
        axis.axhline(level, color=colour, lw=0.6, ls=":", alpha=0.5)
    axis.set_ylim(0, 108); axis.set_xlim(0.5, len(values) + 0.5)
    axis.set_xlabel("residue (concatenated chains)"); axis.set_ylabel("pLDDT")
    axis.set_title(f"{chosen.stem}   per-residue pLDDT", fontsize=10)
    axis.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); plt.show()
    print(f"structure: {chosen}")
    print(f"mean pLDDT {np.mean(values):.1f}   "
          + "   ".join(f"{name} {np.mean(values[int(c - l / 2) : int(c + l / 2)]):.1f}"
                       for c, name, l in labels))

    # --- 3. trajectory loss curve --------------------------------------------
    stem = chosen.stem.split("_seq")[0]
    losses = next((p for p in (CAMPAIGN_FOLDER / "1_Trajectories").glob(f"{stem}*/*_losses.csv")), None)
    if losses is None:
        print("\nNo trajectory losses.csv for this design "
              "(archived, or save_loss_plots was off when it ran).")
    else:
        frame = pd.read_csv(losses)
        numeric = [c for c in frame.columns
                   if pd.api.types.is_numeric_dtype(frame[c]) and frame[c].notna().any()
                   and c.lower() not in ("update", "step", "index", "unnamed: 0")]
        tracked = [c for c in numeric if any(k in c.lower() for k in ("loss", "iptm", "plddt", "ptm", "pae"))][:8] or numeric[:8]
        fig, axis = plt.subplots(figsize=(9, 3.1))
        for column in tracked:
            axis.plot(frame[column].values, lw=1.0, label=column)
        if "stage" in frame.columns:
            changes = frame.index[frame["stage"].ne(frame["stage"].shift())].tolist()[1:]
            for change in changes:
                axis.axvline(change, color="black", lw=0.6, ls="--", alpha=0.4)
        axis.set_xlabel("sequence update"); axis.set_ylabel("value")
        axis.set_title(f"{stem}   gradient-design trajectory", fontsize=10)
        axis.legend(fontsize=7, ncol=2, frameon=False)
        axis.spines[["top", "right"]].set_visible(False)
        fig.tight_layout(); plt.show()


## 10 · Structure viewer

The accepted complexes, best first. Target chains are slate, the binder is rose; `pLDDT` colours by
confidence instead (blue confident, red not — the B-factor column holds pLDDT on a 0–100 scale).

Check the **binding pose**, not just the score: whether the epitope is one the biology can reach,
whether a cropped target would have occluded it in full-length context, and whether a membrane,
glycan or partner protein sits where the binder does.

In [ ]:
#@title Show accepted complexes { display-mode: "form" }
import py3Dmol, gemmi

HOW_MANY      = 4        #@param {type:"integer"}
COLOUR_BY     = "chain"  #@param ["chain", "pLDDT"]
BINDER_CHAINS = 1        #@param {type:"integer"}
#  BINDER_CHAINS: how many chains at the END of the file are the binder - 1 for a normal
#  binder, the number of copies for a homo-oligomer, 2 for an scFv or Fab.

if ACCEPTED is None or not len(ACCEPTED):
    raise SystemExit("Nothing accepted yet.")
ranked_dir = CAMPAIGN_FOLDER / "3_Ranked"
structures = [p for p in sorted(ranked_dir.glob("*.cif")) if "_monomer" not in p.name]
order = [str(d) for d in ACCEPTED.get("design", [])]
by_rank = [p for name in order for p in structures if p.stem.startswith(name)] or structures
shown = by_rank[: max(1, HOW_MANY)]
if not shown:
    raise SystemExit(f"No accepted .cif in {ranked_dir}.")

# The binder is the last chain BC2 writes; targets come first.
columns = min(2, len(shown))
rows = (len(shown) + columns - 1) // columns
view = py3Dmol.view(width=420 * columns, height=360 * rows, viewergrid=(rows, columns))
for index, path in enumerate(shown):
    where = (index // columns, index % columns)
    view.addModel(path.read_text(), "cif", viewer=where)
    if COLOUR_BY == "pLDDT":
        view.setStyle({}, {"cartoon": {"colorscheme":
            {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}}, viewer=where)
    else:
        view.setStyle({}, {"cartoon": {"color": "#5b6b82"}}, viewer=where)
        st = gemmi.read_structure(str(path)); st.setup_entities()
        chain_names = [chain.name for chain in st[0]]
        for binder_chain in chain_names[len(chain_names) - max(1, BINDER_CHAINS):]:
            view.setStyle({"chain": binder_chain}, {"cartoon": {"color": "#c2456b"}}, viewer=where)
    view.zoomTo(viewer=where)
for path in shown:
    row = ACCEPTED[ACCEPTED["design"].astype(str).apply(lambda d: path.stem.startswith(d))]
    score = f"  i_pDAE {row.iloc[0]['i_pDAE']:.3f}" if len(row) and "i_pDAE" in row else ""
    print(f"{path.stem}{score}")
view.show()


## 11 · Export & package

Writes a self-contained bundle beside the run: the ranked table, the accepted complexes and free
binder monomers, the binder sequences as FASTA, a ChimeraX `.cxc` that opens and colours them all,
and `campaign_metadata.json` — the resolved settings, model choices, source revision and checkpoint
hashes. **Keep that file**: it is the only complete record of what produced these designs.

On Drive the bundle is already persisted; off Drive the cell offers it as a download.

In [ ]:
#@title Bundle results { display-mode: "form" }
import tarfile

HOW_MANY_SEQUENCES = 20     #@param {type:"integer"}
INCLUDE_CANDIDATES = False  #@param {type:"boolean"}

if ACCEPTED is None or not len(ACCEPTED):
    raise SystemExit("Nothing accepted yet.")
if "Binder_Sequence" not in ACCEPTED.columns:
    raise SystemExit("No Binder_Sequence column to export.")

BUNDLE = WORK / f"{RUN_NAME}_results"
shutil.rmtree(BUNDLE, ignore_errors=True)
BUNDLE.mkdir(parents=True)

# --- FASTA -------------------------------------------------------------------
NOTE_COLUMNS = ["i_pDAE", "i_pTM", "pLDDT", "i_pAE", "Interface_Residues", "Binder_Mass_kDa", "Binder_pI"]
records = []
for _, row in ACCEPTED.head(HOW_MANY_SEQUENCES).iterrows():
    design = str(row.get("design", "design"))
    notes = " ".join(f"{c}={row[c]}" for c in NOTE_COLUMNS if c in row and pd.notna(row[c]))
    # A multi-chain binder is written in one cell separated by "/", which is not a sequence.
    chains_out = [c for c in str(row["Binder_Sequence"]).split("/") if c]
    for number, sequence in enumerate(chains_out, start=1):
        label = design if len(chains_out) == 1 else f"{design}_chain{number}"
        records.append(f">{label} {notes}\n{sequence}\n")
fasta = BUNDLE / f"{RUN_NAME}_binders.fasta"
fasta.write_text("".join(records))

# --- structures + tables -----------------------------------------------------
structure_dir = BUNDLE / "structures"; structure_dir.mkdir()
copied = []
for path in sorted((CAMPAIGN_FOLDER / "3_Ranked").glob("*.cif")):
    shutil.copy2(path, structure_dir / path.name)
    if "_monomer" not in path.name:
        copied.append(path.name)
for relative in ("3_Ranked/!_Ranked.csv", "2_Refolded/!_Refolded.csv",
                 "1_Trajectories/!_Trajectories.csv", "campaign_metadata.json", "summary.csv"):
    source = CAMPAIGN_FOLDER / relative
    if source.exists():
        shutil.copy2(source, BUNDLE / Path(relative).name)
shutil.copy2(CAMPAIGN_JSON, BUNDLE / "campaign.json")
if INCLUDE_CANDIDATES and (CAMPAIGN_FOLDER / "2_Refolded" / "Complexes").exists():
    shutil.copytree(CAMPAIGN_FOLDER / "2_Refolded" / "Complexes", BUNDLE / "candidates")

# --- ChimeraX session script -------------------------------------------------
script = ["# open this beside the structures/ folder:  chimerax " + f"{RUN_NAME}.cxc", ""]
for number, name in enumerate(copied, start=1):
    script.append(f"open structures/{name}")
script += ["", "hide atoms", "show cartoons", "color #* gray(180)",
           "# the binder is the LAST chain of each model - recolour it, e.g.:",
           "# color /B crimson",
           "# or colour by confidence (B-factors hold pLDDT, 0-100):",
           "# color bfactor palette alphafold",
           "set bgColor white", "lighting soft", "view"]
(BUNDLE / f"{RUN_NAME}.cxc").write_text("\n".join(script) + "\n")

# --- pack --------------------------------------------------------------------
archive = WORK / f"{RUN_NAME}_results.tar.gz"
with tarfile.open(archive, "w:gz") as tar:
    tar.add(BUNDLE, arcname=BUNDLE.name)

print(f"bundle : {BUNDLE}")
print(f"archive: {archive}  ({archive.stat().st_size / 1e6:.1f} MB)")
print(f"  {len(copied)} accepted complexes, {len(records)} sequences in {fasta.name}")
print()
print(fasta.read_text()[:1200])

if not ON_DRIVE:
    from google.colab import files
    files.download(str(archive))
else:
    print(f"\nOn Drive already - nothing to download.")


---
### Notes & next steps

- **Read the rejections, not just the acceptances.** `2_Refolded/!_Refolded.csv` carries
  `failed_filters` per candidate, and step 8 tallies them. A campaign failing overwhelmingly on one
  filter is telling you the objective and the filter disagree — usually the epitope is hard, the
  binder length is wrong, or `forced_targeting` is fighting the surface.
- **`i_pDAE` is not affinity.** A confident pose can still be biologically inaccessible. Before
  ordering anything: inspect the binder against the *full-length* target, not the crop; consider
  glycans, membrane orientation and competing partners; and check `Surface_Hydrophobicity` for
  aggregation risk.
- **The desperation ladder.** `autotuned` in the trajectory table names settings that differed from
  the campaign's own when an attempt ran. An entry naming `initial_guess`, `target_flexibility`,
  `validation_model` or raised `design_recycles` means that attempt ran against an *easier task than
  you asked for* — weigh those designs accordingly.
- **Re-rank and re-filter without redesigning.** From the repo directory:
  `python3 bindcraft.py rank <results folder> --on i_pTM` and
  `python3 bindcraft.py filter <results folder> --where Interface_BuriedArea>=600`.
  `python3 bindcraft.py score <structure.cif>` measures a complex from any source.
- **Throughput.** Colab is for a handful of designs and for working out settings. A real campaign is
  a Slurm array — `sbatch bindcraft.slurm campaign.json` from a clone, with the campaign JSON this
  notebook wrote. `resume` means the two can even share a results folder.
- **Docs:** [README](https://github.com/PacesaLab/BindCraft2) ·
  [design guide](https://github.com/PacesaLab/BindCraft2/blob/main/docs/design-guide.md) ·
  [settings reference](https://github.com/PacesaLab/BindCraft2/blob/main/docs/reference.md) ·
  [outputs and measurements](https://github.com/PacesaLab/BindCraft2/blob/main/docs/outputs.md) ·
  [example campaigns](https://github.com/PacesaLab/BindCraft2/blob/main/examples/README.md)

BindCraft 2 is by the [Pacesa lab](https://github.com/PacesaLab); it builds on AlphaFold 2 (DeepMind),
ColabDesign (Sergey Ovchinnikov), ProteinMPNN (Justas Dauparas) and HyperMPNN. This notebook only
drives it — cite the underlying work.